# Google's Meridian MMM (an improved version of LightweightMMM)
To implement the Marketing Mix Modeling method, Google's Meridian library was utilized. This library was developed as an improvement on Lightweighted MMM, which also offers a hierarchical approach. If state or regional data is available, this geographic-based hierarchical approach can yield more accurate results.


Key model improvements:
- Usage of only the Hill-Adstock architecture
- Automatic scaling of input data
- Usage of knots for trend and seasonality (time-varying intercept approach)
- Ability to use variables unrelated to direct costs (organic_media and non-media treatment)
- Improvement in national and geo-level models by adding a parameter that determines the standard deviation of the channel’s media ratio at different geographical locations (eta_m)
- Model ROI definition (the model directly evaluates ROI as part of its results, reflecting realistic media effects)
- Calibrating MMM with experiments and a priori information (to prevent overestimation or underestimation of channel performance due to noise or model displacement)
- Ability to use reach (number of unique viewers over time) and frequency (average number of displays per viewer) as model inputs
- Ability to use Google Query Volume (GQV) as a control variable to reduce bias and account for organic demand

# Country Deep-Dive (MMM Validation EDA)

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
DATA_PATH = Path("Data_for_MMM.xlsx")
TARGET = "kpi_org"
GEO = "RU"
geo = GEO
start_date = '2022-01-01'

LAGS = [0, 1, 2, 3, 4]
WINDOW_ROLL = 8
MIN_N_CORR = 12
MIN_N_ACTIVE_IN_WINDOW = 6
LAST_MONTHS = 12

df_raw = pd.read_excel(DATA_PATH)
df_raw["time"] = pd.to_datetime(df_raw["time"])
df_raw = df_raw[df_raw.time >= start_date]
df_raw = df_raw[df_raw.geo == GEO]

spend_cols = [c for c in df_raw.columns if c.endswith("_spend")]
SPEND_TO_MODEL = {
    c: c.replace("_spend", "")
    for c in spend_cols
}

df_raw.head()

## 0. Unified visual style + helpers

In [ ]:
# ---------- Unified Visual Style ----------
COLORS = {
    "organics": "tab:blue",
    "spend": "tab:orange",
    "visits": "tab:green",
    "grey": "0.75",
}

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 2.0,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "legend.frameon": False,
})

def add_linear_trend(ax, x, y, color="black", label="Linear trend"):
    x = np.asarray(x); y = np.asarray(y)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0)
    if m.sum() < 5:
        return None

    # y = a + b*x
    b, a = np.polyfit(x[m], y[m], 1)

    xs = np.linspace(x[m].min(), x[m].max(), 100)
    ys = a + b*xs
    ax.plot(xs, ys, color=color, linewidth=2.5, label=label)

    # R^2
    yhat = a + b*x[m]
    ss_res = np.sum((y[m] - yhat) ** 2)
    ss_tot = np.sum((y[m] - np.mean(y[m])) ** 2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return {"a": a, "b": b, "r2": r2, "n": int(m.sum())}


def format_time_axis(ax, interval=3, rotation=0, ha="center"):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    for label in ax.get_xticklabels():
        label.set_rotation(rotation)
        label.set_ha(ha)

def plot_dual_ts(time, y_left, y_right, left_label, right_label, title,
                 left_color=COLORS["organics"], right_color=COLORS["spend"]):
    fig, ax1 = plt.subplots(figsize=(12,4))
    ax1.plot(time, y_left, color=left_color, label=left_label)
    ax1.set_ylabel(left_label, color=left_color)
    ax1.tick_params(axis="y", labelcolor=left_color)

    ax2 = ax1.twinx()
    ax2.plot(time, y_right, color=right_color, label=right_label)
    ax2.set_ylabel(right_label, color=right_color)
    ax2.tick_params(axis="y", labelcolor=right_color)

    ax1.set_title(title)
    format_time_axis(ax1)

    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

def plot_multi_ts(time, series_dict, title, ylabel, top_n=6, min_total=0.0):
    """Readability-first multi-series plot:
    - highlight top_n series by average value
    - render others in grey
    """
    usable = {k:v for k,v in series_dict.items() if np.nansum(np.abs(v)) > min_total}
    if len(usable) == 0:
        print("No series to plot (all are zero / missing).")
        return

    avg = {k: np.nanmean(v) for k,v in usable.items()}
    top = [k for k,_ in sorted(avg.items(), key=lambda x: x[1], reverse=True)[:top_n]]
    others = [k for k in usable.keys() if k not in top]

    fig, ax = plt.subplots(figsize=(12,5))
    for k in others:
        ax.plot(time, usable[k], color=COLORS["grey"], alpha=0.6)
    for k in top:
        ax.plot(time, usable[k], label=k)

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    format_time_axis(ax)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

def scatter_xy_zeroaware(x, y, xlabel, ylabel, title, zero_eps=0.0):
    x = np.asarray(x)
    y = np.asarray(y)

    is_zero = x <= zero_eps
    is_pos = x > zero_eps

    fig, ax = plt.subplots(figsize=(6,4.5))

    # months with zero spend
    if is_zero.any():
        ax.scatter(x[is_zero], y[is_zero], alpha=0.6, marker="x", label=f"x=0 (n={is_zero.sum()})")

    # months with positive spend
    if is_pos.any():
        ax.scatter(x[is_pos], y[is_pos], alpha=0.8, label=f"x>0 (n={is_pos.sum()})")

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


def add_flag_vlines(ax, times):
    for t in times:
        ax.axvline(t, linestyle="--", alpha=0.35)

def scatter_zeroaware_with_trend(x, y, xlabel, ylabel, title):
    x = np.asarray(x); y = np.asarray(y)
    is_zero = np.isfinite(x) & (x <= 0)
    is_pos  = np.isfinite(x) & (x > 0) & np.isfinite(y)

    fig, ax = plt.subplots(figsize=(6,4.5))
    if is_zero.any():
        ax.scatter(x[is_zero], y[is_zero], alpha=0.6, marker="x", label=f"x=0 (n={is_zero.sum()})")
    if is_pos.any():
        ax.scatter(x[is_pos], y[is_pos], alpha=0.8, label=f"x>0 (n={is_pos.sum()})")

    info = add_linear_trend(ax, x, y, color="black", label="Linear trend (x>0)")
    if info:
        ax.set_title(f"{title}\nLinear trend (x>0): R²={info['r2']:.2f}, n={info['n']}")
    else:
        ax.set_title(title + "\nLinear trend: not enough x>0 points")

    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.legend()
    plt.tight_layout()
    plt.show()
    
def plot_corr_heatmap(corr, title):
    corr = corr.round(2)

    fig, ax = plt.subplots(
        figsize=(max(6, 0.55*len(corr.columns)+2), max(5, 0.45*len(corr.index)+2))
    )

    im = ax.imshow(corr.values)  # default colormap
    ax.set_title(title)

    ax.set_xticks(np.arange(len(corr.columns)))
    ax.set_yticks(np.arange(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)

    # значения в ячейках
    vals = corr.values
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            ax.text(j, i, f"{vals[i, j]:.2f}", ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
def corr_and_n(df_sub):
    """Returns (corr_matrix, n_matrix) for df_sub, pairwise for each pair"""
    cols = df_sub.columns
    corr = df_sub.corr(numeric_only=True)
    n = pd.DataFrame(index=cols, columns=cols, dtype=int)

    for i in cols:
        for j in cols:
            n.loc[i, j] = df_sub[[i, j]].dropna().shape[0]
    return corr, n

def plot_heatmap_with_numbers(mat, title, fmt="{:.2f}"):
    mat = mat.copy()
    fig, ax = plt.subplots(figsize=(max(6, 0.55*len(mat.columns)+2), max(5, 0.45*len(mat.index)+2)))
    im = ax.imshow(mat.values)
    ax.set_title(title)

    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticklabels(mat.index)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, fmt.format(mat.values[i, j]), ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
def detect_strong_divergence_periods(
    df_geo,
    time_col="time",
    target_col="kpi_org",
    spend_total_col="spend_total",
    k_persist=2,
    # quantiles (auto thresholds)
    q_org_high=0.80,
    q_org_low=0.20,
    q_spend_low=0.55,
    q_spend_high=0.80,
    q_eff_tail=0.95,
    # dynamics thresholds (still fixed but scale-free)
    spend_jump=0.6,
    org_small=0.05,
    # minimum points to trust quantiles
    min_n=18
):
    work = df_geo[[time_col, target_col, spend_total_col]].copy()
    work = work.sort_values(time_col).reset_index(drop=True)

    # pct changes
    work["spend_total_pct"] = work[spend_total_col].pct_change()
    work["org_pct"] = work[target_col].pct_change()

    # efficiency
    work["org_per_spend"] = np.where(work[spend_total_col] > 0, work[target_col] / work[spend_total_col], np.nan)

    def robust_z(s):
        s = np.asarray(s, dtype=float)
        med = np.nanmedian(s)
        mad = np.nanmedian(np.abs(s - med))
        if mad == 0 or np.isnan(mad):
            return np.full_like(s, np.nan)
        return 0.6745 * (s - med) / mad

    work["org_per_spend_z"] = robust_z(work["prg_per_spend"].values)

    # if too short series, return empty
    if work.shape[0] < min_n:
        work["flag_period_strong_divergence"] = False
        return work, pd.DataFrame()

    # quantile thresholds
    org_hi = work[target_col].quantile(q_org_high)
    org_lo = work[target_col].quantile(q_org_low)
    sp_lo = work[spend_total_col].quantile(q_spend_low)
    sp_hi = work[spend_total_col].quantile(q_spend_high)

    eff_thr = work["org_per_spend_z"].abs().quantile(q_eff_tail)

    # flags
    work["flag_spend_jump_org_flat"] = (work["spend_total_pct"].abs() >= spend_jump) & (work["org_pct"].abs() <= org_small)
    work["flag_eff_outlier"] = work["org_per_spend_z"].abs() >= eff_thr

    work["flag_org_high_spend_low"] = (work[target_col] >= org_hi) & (work[spend_total_col] <= sp_lo)
    work["flag_org_low_spend_high"] = (work[target_col] <= org_lo) & (work[spend_total_col] >= sp_hi)

    # persist helper
    def runs_ge_k(s, k=2):
        return s.rolling(k).sum() >= k

    work["flag_spend_jump_org_flat_persist"] = runs_ge_k(work["flag_spend_jump_org_flat"], k_persist)
    work["flag_eff_outlier_persist"] = runs_ge_k(work["flag_eff_outlier"], k_persist)
    work["flag_org_high_spend_low_persist"] = runs_ge_k(work["flag_org_high_spend_low"], k_persist)
    work["flag_org_low_spend_high_persist"] = runs_ge_k(work["flag_org_low_spend_high"], k_persist)

    work["flag_period_strong_divergence"] = (
        work["flag_spend_jump_org_flat_persist"]
        | work["flag_eff_outlier_persist"]
        | work["flag_org_high_spend_low_persist"]
        | work["flag_org_low_spend_high_persist"]
    )

    flags = work.loc[work["flag_period_strong_divergence"], [
        time_col, target_col, spend_total_col,
        "spend_total_pct", "org_pct",
        "org_per_spend", "org_per_spend_z",
        "flag_spend_jump_org_flat_persist",
        "flag_eff_outlier_persist",
        "flag_org_high_spend_low_persist",
        "flag_org_low_spend_high_persist"
    ]].copy()

    # rounding for readability
    if not flags.empty:
        flags["spend_total_pct"] = flags["spend_total_pct"].round(2)
        flags["org_pct"] = flags["org_pct"].round(2)
        flags["org_per_spend"] = flags["org_per_spend"].round(3)
        flags["org_per_spend_z"] = flags["org_per_spend_z"].round(2)

    return work, flags

def rolling_corr_filtered(x, y, window, min_n=4, active_mask=None):
    """
    Rolling correlation corr(x, y) within a window, computed only on points that pass:
      - finite x and y
      - (optional) active_mask == True
    active_mask: array-like bool, same length as x/y (e.g., y>0).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    out = np.full(n, np.nan)

    if active_mask is None:
        active_mask = np.ones(n, dtype=bool)
    else:
        active_mask = np.asarray(active_mask, dtype=bool)

    for i in range(n):
        start = max(0, i - window + 1)
        idx = np.arange(start, i + 1)

        m = active_mask[idx] & np.isfinite(x[idx]) & np.isfinite(y[idx])
        if m.sum() < min_n:
            continue

        xv = x[idx][m]
        yv = y[idx][m]
        if np.nanstd(xv) == 0 or np.nanstd(yv) == 0:
            continue

        out[i] = np.corrcoef(xv, yv)[0, 1]

    return out


## 0. Data preparation

In [ ]:
df = df_raw.copy()

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

TARGET = "kpi_org"

spend_cols = [c for c in df.columns if c.endswith("_spend")]
control_cols = [c for c in df.columns if c not in ["time","geo",TARGET] + spend_cols]
control_cols = [c for c in control_cols if pd.api.types.is_numeric_dtype(df[c])]

df["spend_total"] = df[spend_cols].sum(axis=1)

# Only months with expenses
mask_active_total = df["spend_total"].fillna(0) > 0

# Creation of the mask_empty_period (where all expenses = 0 and controls are missing (NaN)
mask_controls_present = df[control_cols].notna().any(axis=1) if len(control_cols) > 0 else pd.Series(False, index=df.index)
mask_spend_present = df[spend_cols].fillna(0).sum(axis=1) > 0
mask_empty_period = (~mask_spend_present) & (~mask_controls_present)

# Expenses shares
for c in spend_cols:
    df[c + "_share"] = np.where(df["spend_total"]>0, df[c] / df["spend_total"], np.nan)

df[[ "time","geo",TARGET,"spend_total"] + spend_cols + control_cols].head()

## 1. Time-series: Organics vs Spend, Organics vs Visits, Spend by channel

In [ ]:
plot_dual_ts(
    time=df["time"],
    y_left=df[TARGET],
    y_right=df["spend_total"],
    left_label="Organics",
    right_label="Total Spend",
    title="Organics vs Total Spend (dual axis)"
)

if "Visits" in df.columns:
    plot_dual_ts(
        time=df["time"],
        y_left=df[TARGET],
        y_right=df["Visits"],
        left_label="Organics",
        right_label="Visits",
        title="Organics vs Visits (dual axis)",
        left_color=COLORS["Organics"],
        right_color=COLORS["visits"]
    )

# ---------- Spend by channel (stacked) + Organics line ----------

df_plot = df[["time"] + spend_cols].copy()
df_plot = df_plot.set_index("time")

# убираем каналы с нулевым спендом
non_zero = [c for c in spend_cols if df[c].abs().sum() > 0]
df_plot = df_plot[non_zero]

fig, ax1 = plt.subplots(figsize=(12,5))

# --- stacked area: spend ---
ax1.stackplot(
    df_plot.index,
    [df_plot[c].values for c in df_plot.columns],
    labels=df_plot.columns,
    alpha=0.85
)

ax1.set_ylabel("Spend")
ax1.set_title("Spend by channel (stacked) + Organics")

# --- Organics line on second axis ---
ax2 = ax1.twinx()
ax2.plot(
    df["time"],
    df[TARGET],
    color=COLORS["organics"],
    linewidth=3,
    label="Organics"
)
ax2.set_ylabel("Organics")
ax2.tick_params(axis="y", labelcolor=COLORS["organics"])

# --- legends ---
ax1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Spend channels")
ax2.legend(loc="upper right")

# --- formatting ---
format_time_axis(ax1)
plt.tight_layout()
plt.show()

![](images/Organic_vs_Total_Spend.png)

![](images/Spend_by_Channel_Organic.png)

## 2. Scatter analysis (incl. lags)

In [ ]:
for c in spend_cols:
    if np.nansum(np.abs(df[c].values)) == 0:
        continue
    scatter_zeroaware_with_trend(df[c].values, df[TARGET].values, c, "Organics", f"Organics vs {c}")

![](images/Linear_Regression_MMM.png)

## 3. Correlations

In [ ]:
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
def classify_corr(df, penalty_factor=0.5):
    """
    Classify products into B / C / D based on effectiveness and
    assign Class A to products with all NaN correlations.
    
    Rules:
    - Class A: all correlations NaN
    - Class D: zero effectiveness
    - Class B: top 30% positive effectiveness
    - Class C: next 40% positive effectiveness
    - Ranking applied to B, C, D
    """
    df = df.iloc[1:].copy()
    df_work = df.copy()

    # Identify all-NaN products
    nan_mask = df_work.isna().all(axis=1)

    # Initialize
    df_work["Effectiveness_Score"] = 0.0
    df_work["Class"] = None
    df_work["Rank"] = None

    # Ensure numeric dtype for correlations to avoid warnings
    corr_cols = df.columns
    df_work[corr_cols] = df_work[corr_cols].apply(pd.to_numeric, errors='coerce')

    # Calculate effectiveness for non-NaN products
    for product in df_work[~nan_mask].index:
        row = df_work.loc[product].fillna(0)

        max_positive = row[row > 0].max() if (row > 0).any() else 0
        abs_sum = np.abs(row).sum()
        stability = row[row > 0].sum() / abs_sum if abs_sum != 0 else 0
        min_negative = row[row < 0].min() if (row < 0).any() else 0
        penalty = penalty_factor * abs(min_negative)

        effectiveness = (max_positive * stability) - penalty
        df_work.loc[product, "Effectiveness_Score"] = round(max(0, effectiveness), 6)

    # Class A → all-NaN correlations
    df_work.loc[nan_mask, "Class"] = "A"

    # ---- Percentile classification for non-NaN products ----
    valid_df = df_work[~nan_mask].copy()
    # Sort descending by effectiveness
    valid_df = valid_df.sort_values("Effectiveness_Score", ascending=False)
    n = len(valid_df)

    if n > 0:
        # Top 30% → B, next 40% → C, bottom 30% → D
        top_cut = int(np.ceil(n * 0.30))
        mid_cut = int(np.ceil(n * 0.70))  # 30% + 40%

        # Assign temporary classes
        valid_df.iloc[:top_cut, valid_df.columns.get_loc("Class")] = "B"
        valid_df.iloc[top_cut:mid_cut, valid_df.columns.get_loc("Class")] = "C"
        valid_df.iloc[mid_cut:, valid_df.columns.get_loc("Class")] = "D"

        # Enforce zero-effectiveness → D
        zero_mask = valid_df["Effectiveness_Score"] == 0
        valid_df.loc[zero_mask, "Class"] = "D"

        # Assign ranks only to B, C, D
        rank_mask = valid_df["Class"].isin(["B", "C", "D"])
        valid_df.loc[rank_mask, "Rank"] = range(1, rank_mask.sum() + 1)

    # Merge back
    df_work.update(valid_df)

    # Ensure A products have no rank
    df_work.loc[df_work["Class"] == "A", "Rank"] = None

    # Final sort: B → C → D → A
    class_order = {"B": 1, "C": 2, "D": 3, "A": 4}
    df_work["Class_Order"] = df_work["Class"].map(class_order)

    df_work = df_work.sort_values(
        ["Class_Order", "Effectiveness_Score"],
        ascending=[True, False]
    ).drop(columns="Class_Order")

    return df_work.reset_index()

In [ ]:
def merge_classes(df1, df2):
    """
    Merge product classes from two tables according to business rules.
    
    Both df1 and df2 must have columns: 'Product' and 'Class'
    """
    
    # Merge the two tables on Product
    merged = df1[['index', 'Class']].merge(
        df2[['index', 'Class']],
        on='index',
        how='outer',
        suffixes=('_1', '_2')
    )
    
    def resolve_class(row):
        c1 = row['Class_1']
        c2 = row['Class_2']
        
        # If one is NaN, take the other
        if pd.isna(c1):
            return c2
        if pd.isna(c2):
            return c1
        
        # If classes are equal, keep
        if c1 == c2:
            return c1
        
        # Handle A: take non-A
        if c1 == 'A':
            return c2
        if c2 == 'A':
            return c1
        
        # Mapping for B/C/D transitions (less favorable)
        transition_map = {
            ('B', 'C'): 'C',
            ('C', 'B'): 'C',
            ('C', 'D'): 'C',
            ('D', 'C'): 'D',
            ('B', 'D'): 'D',
            ('D', 'B'): 'D'
        }
        
        return transition_map.get((c1, c2), c2)  # default to df2 if not in map
    
    merged['Class_Final'] = merged.apply(resolve_class, axis=1)
    class_order = {"B": 1, "C": 2, "D": 3, "A": 4}
    merged["Class_Order"] = merged["Class_Final"].map(class_order)
    merged = merged.sort_values(
        ["Class_Order"],
        ascending=[True]
    ).drop(columns="Class_Order")
    
    return merged[['index', 'Class_Final']]

### correlations with lags over the last year 

In [ ]:
# --- Prepare data for correlation analysis ---
# Assumption: 0 in spend / controls ≈ missing data

corr_spend_cols = spend_cols
corr_control_cols = control_cols

df_corr = df.copy()
cutoff = pd.Timestamp.now().normalize().replace(day=1) - pd.DateOffset(months=12)
df_corr = df_corr[df_corr['time'] >= cutoff]

# Replace 0 -> NaN for spend
df_corr[corr_spend_cols] = df_corr[corr_spend_cols].replace(0, np.nan)

# Replace 0 -> NaN for controls
if len(corr_control_cols) > 0:
    df_corr[corr_control_cols] = df_corr[corr_control_cols].replace(0, np.nan)

# Active months = есть хотя бы один НЕ-NA spend
mask_active_total = df_corr[corr_spend_cols].notna().any(axis=1)

# Полностью пустые периоды
mask_empty_period = (
    df_corr[corr_spend_cols].isna().all(axis=1)
    & (df_corr[corr_control_cols].isna().all(axis=1) if len(corr_control_cols) > 0 else True)
)

In [ ]:
lag_corr = []
for lag in [-4, -3,-2,-1, 0]:
    tmp = df_corr[[TARGET, "spend_total"] + spend_cols].copy()
    tmp[TARGET] = tmp[TARGET].shift(lag)
    lag_corr.append(tmp.corr(numeric_only=True)[TARGET].rename(f"Organics_t+{lag}"))

lag_corr_tbl_one_year = (
    pd.concat(lag_corr, axis=1)
    .loc[["spend_total"] + spend_cols]
    .round(2)
)
lag_corr_tbl_one_year

In [ ]:
class_corr_one_year = classify_corr(lag_corr_tbl_one_year)
class_corr_one_year

### correlations over all period, lags

In [ ]:
# --- Prepare data for correlation analysis ---
# Assumption: 0 in spend / controls ≈ missing data

corr_spend_cols = spend_cols
corr_control_cols = control_cols

df_corr = df.copy()

# Replace 0 -> NaN for spend
df_corr[corr_spend_cols] = df_corr[corr_spend_cols].replace(0, np.nan)

# Replace 0 -> NaN for controls
if len(corr_control_cols) > 0:
    df_corr[corr_control_cols] = df_corr[corr_control_cols].replace(0, np.nan)

In [ ]:
# Active months = есть хотя бы один НЕ-NA spend
mask_active_total = df_corr[corr_spend_cols].notna().any(axis=1)

# Полностью пустые периоды
mask_empty_period = (
    df_corr[corr_spend_cols].isna().all(axis=1)
    & (df_corr[corr_control_cols].isna().all(axis=1) if len(corr_control_cols) > 0 else True)
)

In [ ]:
lag_corr = []
for lag in [-4, -3,-2,-1,0]:
    tmp = df[[TARGET, "spend_total"] + spend_cols].copy()
    tmp[TARGET] = tmp[TARGET].shift(lag)
    lag_corr.append(tmp.corr(numeric_only=True)[TARGET].rename(f"Organics_t+{lag}"))

lag_corr_tbl = (
    pd.concat(lag_corr, axis=1)
    .loc[["spend_total"] + spend_cols]
    .round(2)
)
lag_corr_tbl

In [ ]:
class_corr = classify_corr(lag_corr_tbl)
class_corr

### final classes according to correlation

In [ ]:
final_class_corr = merge_classes(class_corr, class_corr_one_year)
final_class_corr

### Spend ↔ Spend correlation & Spend ↔ Controls correlation

In [ ]:
df_spend_corr = df_corr[corr_spend_cols]

# active months
df_spend_active = df_spend_corr.loc[mask_active_total]
corr_sa, n_sa = corr_and_n(df_spend_active)

plot_heatmap_with_numbers(corr_sa.round(2), "Spend ↔ Spend corr (активные месяцы)", fmt="{:.2f}")
plot_heatmap_with_numbers(n_sa, "Spend ↔ Spend N (кол-во активных месяцев)", fmt="{:.0f}")


In [ ]:
if len(corr_control_cols) > 0:
    df_sc_corr = df_corr[corr_spend_cols + corr_control_cols]
    df_sc = df[spend_cols + control_cols]

    # active months
    df_sc_active = df_sc_corr.loc[mask_active_total]
    corr_sca, n_sca = corr_and_n(df_sc_active)
    plot_heatmap_with_numbers(
        corr_sca.loc[corr_spend_cols, corr_control_cols].round(2),
        "Spend ↔ Controls corr (0→NA, active months)",
        fmt="{:.2f}"
    )
    plot_heatmap_with_numbers(
        n_sca.loc[corr_spend_cols, corr_control_cols],
        "Spend ↔ Controls N (0→NA, active months)",
        fmt="{:.0f}"
    )


### 4. VIF по spend каналам

Оценка мультиколлинеарности между spend-каналами. VIF считается по активным месяцам
(где spend > 0 по каждому из каналов, участвующих в регрессии).

VIF канала должен быть меньше 5


In [ ]:
def calc_vif(x, min_n=MIN_N_CORR):
    """Return VIF for each column in x (DataFrame)."""
    out = []
    for col in x.columns:
        y = x[col].astype(float).values
        others = x.drop(columns=[col])
        if others.shape[1] == 0:
            out.append((col, np.nan, 0))
            continue

        X = others.astype(float).values
        mask = np.isfinite(y) & np.isfinite(X).all(axis=1)
        if mask.sum() < min_n:
            out.append((col, np.nan, int(mask.sum())))
            continue

        yv = y[mask]
        Xv = X[mask]
        if np.nanstd(yv) == 0:
            out.append((col, np.nan, int(mask.sum())))
            continue

        Xv = np.column_stack([np.ones(len(Xv)), Xv])
        coef, *_ = np.linalg.lstsq(Xv, yv, rcond=None)
        yhat = Xv @ coef
        ss_res = np.sum((yv - yhat) ** 2)
        ss_tot = np.sum((yv - np.mean(yv)) ** 2)
        if ss_tot == 0:
            out.append((col, np.nan, int(mask.sum())))
            continue

        r2 = 1 - ss_res / ss_tot
        if r2 >= 1:
            vif = np.inf
        else:
            vif = 1 / (1 - r2)
        out.append((col, float(vif), int(mask.sum())))

    return pd.DataFrame(out, columns=["spend_col", "vif", "n_used"])

# Use GEO slice if dfk is not defined
try:
    df_vif = df_spend_corr.copy()
except NameError:
    df_vif = df[df["geo"] == GEO].copy()
    df_vif["time"] = pd.to_datetime(df_vif["time"])
    df_vif = df_vif.sort_values("time").reset_index(drop=True)

spend_cols = [c for c in df_vif.columns if c.endswith("_spend")]

# active months per channel are enforced inside pairwise regression
x = df_vif[spend_cols].fillna(0.0)

vif_tbl = calc_vif(x)

vif_tbl["vif"] = vif_tbl["vif"].round(2)
vif_tbl = vif_tbl.sort_values("vif", ascending=False)

vif_tbl

## 5. Pre-model channel decomposition checks

In [ ]:
SORT_BY = "best_corr_abs"   # "best_corr_abs", "corr_lag1_active", "corr_lag1", "sum_share_total", "avg_spend"
MIN_N = MIN_N_CORR       # минимум наблюдений для corr (иначе NaN)

def corr_org_lead_vs_spend_with_n(df, target, spend_col, lag, active_only=False):
    """
    corr( Organics_{t+lag}, Spend_t ) + n used.
    active_only=True: используем только строки, где Spend_t > 0.
    """
    tmp = df[[spend_col, target]].copy()
    tmp[target] = tmp[target].shift(-lag)

    if active_only:
        tmp = tmp[tmp[spend_col].fillna(0) > 0]

    tmp = tmp.dropna()
    n = len(tmp)
    if n < MIN_N:
        return np.nan, n

    if tmp[spend_col].std() == 0 or tmp[target].std() == 0:
        return np.nan, n

    v = tmp.corr(numeric_only=True).iloc[0, 1]
    return v, n

def best_corr_and_lag(row, corr_cols, lag_from_col):
    """
    Возвращает (best_col, best_abs, best_lag) с защитой от случая "все NaN".
    """
    vals = row[corr_cols]
    if vals.isna().all():
        return np.nan, np.nan, np.nan
    best_col = vals.abs().idxmax()
    best_abs = float(abs(vals[best_col]))
    best_lag = lag_from_col(best_col)
    return best_col, best_abs, best_lag

# ---------- 1) Base activity / coverage / totals ----------
n_months = len(df)

base = pd.DataFrame({"channel": spend_cols})
base["n_months"] = n_months
base["n_pos"] = [(df[c].fillna(0) > 0).sum() for c in spend_cols]
base["pos_share"] = base["n_pos"] / n_months

base["sum_spend"] = [df[c].sum() for c in spend_cols]
base["avg_spend"] = [df[c].mean() for c in spend_cols]
base["median_spend"] = [df[c].median() for c in spend_cols]

total_sum = base["sum_spend"].sum()
base["sum_share_total"] = np.where(total_sum > 0, base["sum_spend"] / total_sum, np.nan)

# ---------- 2) Share stats ----------
share_cols = [c + "_share" for c in spend_cols]
share_stats = (
    df[share_cols]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .T[["mean", "50%", "std", "min", "25%", "75%", "max"]]
    .rename(columns={"50%": "median", "25%": "p25", "75%": "p75"})
)
share_stats.index = [c.replace("_share", "") for c in share_stats.index]
share_stats = share_stats.add_prefix("share_")

# ---------- 3) Corr by lags (+ n used) ----------
proxy = base[["channel"]].copy()

for lag in LAGS:
    col_all = "corr_same_period" if lag == 0 else f"corr_lag{lag}"
    n_all   = "n_same_period" if lag == 0 else f"n_lag{lag}"

    col_act = "corr_same_period_active" if lag == 0 else f"corr_lag{lag}_active"
    n_act   = "n_same_period_active" if lag == 0 else f"n_lag{lag}_active"

    vals_all, ns_all, vals_act, ns_act = [], [], [], []

    for c in spend_cols:
        v1, n1 = corr_org_lead_vs_spend_with_n(df, TARGET, c, lag, active_only=False)
        v2, n2 = corr_org_lead_vs_spend_with_n(df, TARGET, c, lag, active_only=True)

        vals_all.append(v1); ns_all.append(n1)
        vals_act.append(v2); ns_act.append(n2)

    proxy[col_all] = vals_all
    proxy[n_all] = ns_all
    proxy[col_act] = vals_act
    proxy[n_act] = ns_act

corr_cols_all = ["corr_same_period"] + [f"corr_lag{l}" for l in LAGS if l != 0]
corr_cols_act = ["corr_same_period_active"] + [f"corr_lag{l}_active" for l in LAGS if l != 0]

def lag_from_col_all(s):
    if s == "corr_same_period":
        return 0
    return int(s.replace("corr_lag", ""))

def lag_from_col_act(s):
    if s == "corr_same_period_active":
        return 0
    return int(s.replace("corr_lag", "").replace("_active", ""))

# best by ALL
best_all = proxy.apply(lambda r: best_corr_and_lag(r, corr_cols_all, lag_from_col_all), axis=1, result_type="expand")
best_all.columns = ["best_corr_col_all", "best_corr_abs_all", "best_lag_all"]
proxy = pd.concat([proxy, best_all], axis=1)

# best by ACTIVE (fallback может быть NaN)
best_act = proxy.apply(lambda r: best_corr_and_lag(r, corr_cols_act, lag_from_col_act), axis=1, result_type="expand")
best_act.columns = ["best_corr_col_active", "best_corr_abs_active", "best_lag_active"]
proxy = pd.concat([proxy, best_act], axis=1)

# unified best (предпочитаем active, иначе all)
proxy["best_corr_col"] = proxy["best_corr_col_active"].where(proxy["best_corr_abs_active"].notna(), proxy["best_corr_col_all"])
proxy["best_corr_abs"] = proxy["best_corr_abs_active"].where(proxy["best_corr_abs_active"].notna(), proxy["best_corr_abs_all"])
proxy["best_lag"] = proxy["best_lag_active"].where(proxy["best_corr_abs_active"].notna(), proxy["best_lag_all"])

# ---------- 4) Merge all together ----------
diag = (
    base.set_index("channel")
    .join(share_stats, how="left")
    .join(proxy.set_index("channel"), how="left")
    .reset_index()
)

# ---------- 5) Quick flags / labels ----------
diag["flag_sparse"] = diag["pos_share"] < 0.25

# для low-corr используем active lag1 если он есть, иначе all lag1
corr_lag1_pref = diag["corr_lag1_active"].where(diag["corr_lag1_active"].notna(), diag["corr_lag1"])
diag["flag_big_spend_low_corr"] = (diag["sum_share_total"] >= 0.15) & (corr_lag1_pref.abs() < 0.1)

diag["channel_type"] = np.select(
    [diag["flag_sparse"], diag["flag_big_spend_low_corr"]],
    ["sparse/episodic", "high spend + low corr"],
    default="regular"
)

# ---------- 6) Sorting + rounding ----------
# если SORT_BY нет — fallback на best_corr_abs
if SORT_BY not in diag.columns:
    SORT_BY = "best_corr_abs"

diag = diag.sort_values(SORT_BY, ascending=False)

diag_display = diag.copy()

# доли
for c in ["pos_share", "sum_share_total"] + [x for x in diag_display.columns if x.startswith("share_")]:
    diag_display[c] = diag_display[c].round(3)

# корреляции
for c in [x for x in diag_display.columns if x.startswith("corr_")] + ["best_corr_abs", "best_corr_abs_active", "best_corr_abs_all"]:
    if c in diag_display.columns:
        diag_display[c] = diag_display[c].round(2)

# деньги
for c in ["sum_spend", "avg_spend", "median_spend"]:
    diag_display[c] = diag_display[c].round(0)

cols_main = [
    "channel", "channel_type",
    "sum_share_total", "sum_spend", "avg_spend",
    "share_mean", "share_std",

    # ALL
    #"corr_same_period", "corr_lag1", "corr_lag2", "corr_lag3", "corr_lag4",

    # ACTIVE-only
    "corr_same_period_active", "corr_lag1_active", "corr_lag2_active", "corr_lag3_active", "corr_lag4_active",

    # best (unified + components)
    #"best_lag", "best_corr_abs", "best_corr_col",
    "best_lag_active", "best_corr_abs_active",
    #"best_lag_all", "best_corr_abs_all",

    # Ns
    #"n_same_period", "n_lag1", "n_lag2", "n_lag3", "n_lag4",
    "n_same_period_active", #"n_lag1_active", "n_lag2_active", "n_lag3_active", "n_lag4_active",
]
cols_main = [c for c in cols_main if c in diag_display.columns]

diag_display[cols_main]


# MMM

Список выходных диагностик и графиков:

model_fit = visualizer.ModelFit(pairs[('RU',1)][1])
model_fit.plot_model_fit(include_baseline=True,
                         include_ci=True)

tt = visualizer.MediaSummary(pairs[('RU',1)][1])
tt.plot_channel_contribution_area_chart()

reviewer.ModelReviewer(pairs[('RU',1)][1]).run()

tt.plot_spend_vs_contribution()

visualizer.MediaEffects(pairs[('RU',1)][1],use_kpi=True).plot_response_curves()


In [ ]:
import arviz as az
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder
from meridian.data import test_utils
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.model import knots
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp
from meridian.backend import tfd
from matplotlib import pyplot as plt
from itertools import chain, combinations
from collections import defaultdict
from meridian.analysis.review import reviewer
import statsmodels.api as sm

import time

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

def stl_components(y: np.ndarray, period=12):
    res = sm.tsa.STL(np.asarray(y, float), period=period, robust=True).fit()
    return res.trend, res.seasonal

def local_extrema_idx(x: np.ndarray, min_prominence: float = 0.25):
    x = np.asarray(x, float)
    if len(x) < 5:
        return np.array([], dtype=int)
    thr = min_prominence * (np.max(np.abs(x)) + 1e-9)
    out = []
    for i in range(1, len(x)-1):
        peak = x[i] > x[i-1] and x[i] > x[i+1]
        trough = x[i] < x[i-1] and x[i] < x[i+1]
        if peak or trough:
            prom = min(abs(x[i]-x[i-1]), abs(x[i]-x[i+1]))
            if prom >= thr:
                out.append(i)
    return np.array(out, dtype=int)

def knee_points_from_trend(trend: np.ndarray, top_n: int = 4, exclude_edges: int = 2):
    t = np.asarray(trend, float)
    if len(t) < 6:
        return np.array([], dtype=int)
    d2 = np.abs(np.diff(t, n=2))  # длина T-2
    valid = np.arange(len(d2))
    valid = valid[(valid >= exclude_edges) & (valid <= len(d2)-1-exclude_edges)]
    if len(valid) == 0:
        return np.array([], dtype=int)
    chosen = valid[np.argsort(-d2[valid])[:top_n]] + 1
    return np.sort(chosen)

def select_knots_auto(sphere_interest, product_interest, K_min=6, K_max=10, period=12, season_prominence=0.25, trend_knees=4):
    T = len(sphere_interest)
    trend_b, _ = stl_components(product_interest, period=period)
    _, season_c = stl_components(sphere_interest, period=period)
    season_scaled = season_c / (np.max(np.abs(season_c)) + 1e-9)

    cand = set(local_extrema_idx(season_scaled, season_prominence).tolist())
    cand |= set(knee_points_from_trend(trend_b, top_n=trend_knees).tolist())
    cand |= {0, T-1}

    cand = np.array(sorted(cand), dtype=int)

    if len(cand) < K_min:
        cand = np.array(sorted(set(cand.tolist()) | set(np.linspace(0, T-1, K_min, dtype=int).tolist())), dtype=int)

    if len(cand) > K_max:
        # скоринг (сезонность + переломы)
        d2 = np.abs(np.diff(trend_b, n=2))
        def knee_score(i):
            if i in (0, T-1): return 1e9
            if 1 <= i <= T-2: return d2[i-1]
            return 0.0
        scores = np.array([abs(season_scaled[i]) + 0.7*knee_score(i) for i in cand])
        keep = np.argsort(-scores)[:K_max]
        cand = np.sort(cand[keep])

    return cand

def sanitize_knot_locations(knot_locations, T: int):
    """Meridian-friendly: list[int], уникальные, отсортированные, в диапазоне."""
    kl = [int(x) for x in knot_locations]
    kl = sorted(set(kl))
    if kl[0] != 0:
        kl = [0] + kl
    if kl[-1] != T - 1:
        kl = kl + [T - 1]
    kl = [k for k in kl if 0 <= k <= T-1]
    return kl

# ---------- main: geo-aware ----------
def get_knots_and_priors_for_geo(
    geo: str,
    df_cut: pd.DataFrame,
    *,
    x: float,
    time_col="time",
    sphere_interest_col="sphere_interest",
    product_interest_col="product_interest",
    K_min=6,
    K_max=10,
    period=12,
    eps_min=0.05,
    growth=0.35,
    amp=0.12,
    sigma=0.25,
    manual_knots: dict | None = None,
):
    """
    Возвращает (knot_locations_list, mu_k_list, sigma_k_list).
    manual_knots: dict GEO->list[int] (0-based)
    """
    manual_knots = manual_knots or MANUAL_KNOTS

    df = df_cut.sort_values(time_col).reset_index(drop=True)
    sphere_interest = df[sphere_interest_col].astype(float).to_numpy()
    product_interest = df[product_interest_col].astype(float).to_numpy()
    T = len(df)

    # knots: manual override или auto
    if geo in manual_knots and manual_knots[geo]:
        knot_locations = sanitize_knot_locations(manual_knots[geo], T)
    else:
        knot_locations = select_knots_auto(
            sphere_interest=sphere_interest,
            product_interest=product_interest,
            K_min=K_min,
            K_max=K_max,
            period=period
        )
        knot_locations = sanitize_knot_locations(knot_locations.tolist(), T)

    # STL компоненты (для mu_k)
    trend_b, _ = stl_components(product_interest, period=period)
    _, season_c = stl_components(sphere_interest, period=period)

    trend_norm = (trend_b - trend_b.min()) / (trend_b.max() - trend_b.min() + 1e-9)
    season_scaled = season_c / (np.max(np.abs(season_c)) + 1e-9)

    idx = np.array(knot_locations, dtype=int)

    mu_minus_x = eps_min + growth * trend_norm[idx] + amp * season_scaled[idx]
    mu_k = x + mu_minus_x
    mu_k = np.maximum(mu_k, x + 0.02)   # hard floor для truncatednormal

    K = len(knot_locations)
    sigma_k = np.full(K, sigma, dtype=float)

    return knot_locations, mu_k.tolist(), sigma_k.tolist()

In [ ]:
df = df_raw

geo_list  =['RU'] 

K_LIST = [1]

END_DATES = [1]

spend_channels = [c for c in df_raw.columns if c.endswith('_spend')]
channels_all = [c.replace('_spend', '') for c in spend_channels]

roi_mu    = np.ones(len(spend_channels), dtype=np.float32) #all priors are 1, prior can be changed for any media roi_mu[index] = 0.5
roi_sigma = np.ones(len(spend_channels), dtype=np.float32)

pairs = defaultdict(dict)          # для хранения моделей
results = pd.DataFrame()
# --- утилиты ---

def pick_active_channels(df_geo, channels):
    active = []
    for ch in channels:
        sc = f"{ch}_spend"
        if sc in df_geo.columns and df_geo[sc].sum() > 0:
            active.append(ch)
    return active

for geo in geo_list:
    for start_date in K_LIST:             
        for ml in END_DATES:
            results_rows = pd.DataFrame()
            
            # 1) срез данных по GEO и периоду [start_date, end_date]
            df_geo = df[df['geo'] == geo]
            df_cut = (
                df_geo.loc[(df_geo['time'] >=  pd.Timestamp("2022-01-01"))]
                .reset_index(drop=True)
                .fillna(0)
            )
            if df_cut.empty:
                print(f"{geo} |  нет данных, пропуск")
                continue

            # 2) KPI / cost / priors
            kpi         = df_cut.kpi_org.sum()
            cost        = df_cut[[c for c in df_cut.columns if c.endswith('_spend')]].sum().sum()
            p_mean   = 0.7 # чем выше тем больше ROI раздаем в среднем на все каналы (если большой, то baseline в минус)
            p_sd     = 0.3 # чем шире, тем больше ROI варируется от канала и от времени
            roi_mean = (p_mean * kpi / np.sum(cost))
            roi_sd = (p_sd * kpi / np.sqrt(np.sum(np.power(cost, 2))))

            # 3) активные каналы и контролы
            active_channels = pick_active_channels(df_cut, channels_all)
            if not active_channels:
                print(f"{geo} | нет активных каналов (spend=0)")
                continue

            controls        = ['hol_days']
            media_cols       = [f"{ch}_spend" for ch in active_channels]
            media_spend_cols = [f"{ch}_spend"      for ch in active_channels]

            # 4) билдер input_data
            builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
                kpi_type='non_revenue',
                default_kpi_column='kpi_org',
                default_time_column='time'
            ).with_kpi(df_cut)

            if controls:
                builder = builder.with_controls(df_cut, control_cols=controls)

            builder = builder.with_media(
                df_cut,
                media_cols=media_cols,
                media_spend_cols=media_spend_cols,
                media_channels=active_channels
            )
            input_data = builder.build()

            # 5) приоры по активным каналам
            idx = [channels_all.index(ch) for ch in active_channels]
            roi_mu_sel = roi_mu[idx]
            
            x = float(-np.mean(df_cut.kpi_org) / df_cut.kpi_org.std(ddof=1))
            
            knot_locations, mu_k, sigma_k = get_knots_and_priors_for_geo(
                geo,
                df_cut,
                x=x,
                eps_min=0.05,
                growth=0.6,
                amp=0.15,
                sigma=0.5,
                manual_knots={
                    "RU": [0, 23, 24, 33, 39, 41, 45, 47, 49], #2022-01, according to local minimums and maximums of product_interest
                },
            )
            lower_bound = x + 0.02

            prior = prior_distribution.PriorDistribution(
                roi_m = prior_distribution.lognormal_dist_from_mean_std(
                              np.array(roi_mu_sel * roi_mean, dtype=np.float32),
                              np.array(roi_mu_sel * roi_sd,   dtype=np.float32)),
                knot_values = tfd.TruncatedNormal(mu_k,sigma_k,lower_bound,5),
                gamma_c = tfd.Normal(0.05,0.01)   
            )
            
            model_spec = spec.ModelSpec(
                prior=prior,
                max_lag=4,
                #adstock_decay_spec=adstock_decay_spec,
                media_prior_type='roi',
                knots = knot_locations,
                #enable_aks=True,
            )

            mmm = model.Meridian(input_data=input_data, model_spec=model_spec)
            mmm.sample_prior(1000, seed=0)
            mmm.sample_posterior(
                    n_chains=2, n_adapt=1000, n_burnin=2000, n_keep=1000, seed=0
                )

            # --- 6) метрики: neg baseline prob, R2, wMAPE ---
            a = analyzer.Analyzer(mmm)
            neg_prob = float(a.negative_baseline_probability())

            md  = visualizer.ModelDiagnostics(mmm)
            acc = md.predictive_accuracy_table()
            if 'evaluation_set' in acc.columns:
                    acc = acc[acc['evaluation_set'] == 'All Data']

            def get_metric(df_acc, name):
                    mask = df_acc['metric'] == name
                    return float(df_acc.loc[mask, 'value'].iloc[0]) if mask.any() else np.nan

            r2    = get_metric(acc, 'R_Squared')
            mape  = get_metric(acc, 'MAPE')
            wmape = get_metric(acc, 'wMAPE')

                # сохраняем модель и метрики в память
            pairs[(geo, start_date)][ml] = mmm

            results_rows["geo"] = [geo]
            results_rows["start_date"] = [start_date]
            results_rows["end_date"] = [ml]
            results_rows["n_obs"] = [len(df_cut)]
            results_rows["negative_baseline_prob"] = [neg_prob]
            results_rows["R2"] = [r2]
            results_rows["MAPE"] = [mape]
            results_rows["wMAPE"] = [wmape]

            # --- 7) технические параметры ---
            result = reviewer.ModelReviewer(mmm).run()
            print(result)
            for check_name, check  in zip(result.checks_status, result.results):
                case = getattr(check, 'case', 'no case')
                recommendation = getattr(check, 'recommendation', 'No recommendation available')  # Safe access to the attribute
                results_rows[check_name] = recommendation
                results_rows[check_name + '_res'] = case.name
            
            # --- 8) ПРОМЕЖУТОЧНОЕ СОХРАНЕНИЕ В ФАЙЛ ---
            results_df = pd.DataFrame(results_rows)
            results = pd.concat([results, results_rows], ignore_index = True).fillna(0)

            print(f"✓ {geo} | {start_date} | {ml} "
                      f"| R2={r2:.3f}, wMAPE={wmape:.3f}, neg_p={neg_prob:.3f}")

results.to_csv(f"model_quality_results_{geo}.csv", index=False)
print(f"\nИтог: моделей обучено {len(results_rows)}")

## Model Quality Checks

### MCMC Consistency Check, R-hat
##### R-hat < 1.2
In the Bayesian model (Meridian, LMMM, etc.) parameters are not considered a formula but are selected from the posterior distribution using the MCMC algorithm (usually NUTS/HMC):
starts with several chains, let's say 4;
Each chain starts from a different point of the parameter space
chain step by step "walk" through the space and gradually begin to fill the area where the real posterior lies.

Similarity means that:
all chains have already "forgotten" the starting points;
walk along the same distribution;
their distributions are similar and have stabilized.
Until then, any estimates of ROI, contribution, baseline, etc. can be just an artifact of early iterations.

\begin{align}
\hat{R} = \sqrt{\frac{\hat{V}}{{W}}},
\end{align}

\begin{align}
\hat{V} = \frac{n - 1}{n} W + \frac{1}{n} B,
\end{align}

\begin{align}
B = \frac{\sum_{i=1}^{m} (\bar{x}_i - \bar{x}_{Grand})^2}{m - 1},
\end{align}

\begin{align}
W = \frac{\sum_{i=1}^{m} {s}_i^2}{m}
\end{align}
    where: 
    <br>
    B -  dispersion between chains,
    <br>
    W - intra-chain dispersion,
    <br>
    n - number of iterations in a chain,
    <br>
    m - number of chains.

### Baseline Check, the posterior probability that the baseline is negative
##### p-value < 0.5
The probability of negative baseline. In our case, there can be no negative baseline because there can be no negative value for the number of organic consumers. 
Baseline is the number of organic consumers at zero marketing costs, i.e. people who have become our consumers by chance (not through advertising and not at the expense of the brand name; all organic consumers who come to us WITHOUT interaction with any media).
$$
\text{negative\_baseline\_probability} = \frac{\text{Number of samples where } \beta_0 < 0}{\text{Total number of samples}},
\beta\_0-baseline$$


### BayesianPPP Check (Bayesian Posterior Predictive P-value), posterior predictive p-value
##### 0.5 < p-value < 0.95
Bayesian test of the model’s predictive adequacy. An additional check (is needed to understand model forecast)

- The posterior predictive distribution is constructed by total outcome (total number of organic consumers) and compared with the actual values.
- The posterior predictive p-value is computed; that is the share of simulations where the model gives a more extreme value than the observed one.

Interpretation:
- Value about 0.5 - usually ok, meaning the model can represent data.
- Very small (<0.05) or very large (>0.95) p-value is a sign that the model systematically underestimates/overestimates the outcome.
    - Very large (p-value>0.95) - systematically predicted number of organic consumers more than actual.
    - Very small (p-value<0.05) - systematically predicted number of organic consumers less than actual.
$$
\text{PPP} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{I} \left( \left| \text{expected outcome}_i - \mu_{\text{expected}} \right| \geq \left| \text{actual outcome} - \mu_{\text{expected}} \right| \right)
$$

where:
<br>
$\text{outcome}$ - total number of organic consumers,
<br>
$\text{N}$ - number of predicted outcomes,
<br>
$\mu_{\text{expected}}$ - average predicted outcome,
<br>
$\mathbb{I}$ - indicator function is 1 if the condition is true and 0 otherwise.

### GoodnessOfFit Check, R-squared (R²), MAPE, wMAPE
R-squared (R²) is the percentage of outcome dispersion explained by the model
<br>
MAPE - Mean Absolute Percentage Error
<br>
wMAPE - Weighted MAPE (weighted by volume)

##### R²:
0.8 <= R² <= 1 : Excellent quality
<br>
0.5 <= R² <= 0.7 : Medium quality, audit area
<br>
R² <= 0.5 : Poor quality, model revision
<br>
##### MAPE/wMAPE:
MAPE/wMAPE < 25: Excellent quality
<br>
MAPE/wMAPE < 40 and MAPE > 25: Medium quality
<br>
MAPE/wMAPE > 40 : Poor quality, model review
<br>

$$
\begin{align}
R^2 &= \frac{\sum_{t=1}^{T} (\hat{y}_t - \bar{y})^2}{\sum_{t=1}^{T} (y_t - \bar{y})^2} \\[0.3cm]
\text{MAPE} &= \frac{1}{T} \sum_{t=1}^{T} \frac{|y_t - \hat{y}_t|}{|y_t|} \times 100 \\[0.3cm]
\text{WMAPE} &= \frac{\sum w_i \cdot |y_i|}{\sum w_i \cdot |y_i - \hat{y_i}|}
\end{align}
$$
where:
<br>
${\hat{y}_t}$ - predicted number of organic consumers,
<br>
${\bar{y}_t}$ - average number of organic consumers,
<br>
${{y}_t}$ - actual number of organic consumers.

### PriorPosteriorShift Check
##### The channel posterior on parameters does not deviate from previously set channels prior
Check how far posterior deviates from the prior set. If, at a reasonable amount of data, the posterior doesn’t move much from prior to any channel, then:
- Or the prior is too "hard" (dispersion is too small), and the data simply cannot move it;
- Or there is almost no signal on this channel in the data (noise, small amount of spending, or repetitive behavior).

Average, median, Q1 (quantile 25%), Q3 (quantile 75%) are compared. If there is no statistically significant difference in metrics, then the posterior does not move significantly from prior.

### ROIConsistency Check
##### (posterior_means < prior_roi_los) or (posterior_means > prior_roi_his)
Verify that the ROI posterior is consistent with the specified ROI priors.
The posterior ROI distribution of channels is compared with ranges and prior ROI forms that have been established. (ROI = number of organic consumers / amount of spending)

The channels where the average ROI posterior is less than the lower ROI prior quantile or greater than the upper ROI prior quantile are defined. This indicates a discrepancy between the posterior ROI and the specified ROI prior.
<br>
<center>low_roi_channels = all_channels[posterior_means < prior_roi_los]</center> 
<center> high_roi_channels = all_channels[posterior_means > prior_roi_his]</center> 

where:<br>
posterior_means - average ROI posterior, <br>
prior_roi_los - lower quantile of the ROI prior, <br>
prior_roi_his - higher quantile of the ROI prior.

In [ ]:
reviewer.ModelReviewer(pairs[(geo,1)][1]).run()

In [ ]:
model_fit = visualizer.ModelFit(pairs[(geo,1)][1])
model_fit.plot_model_fit(include_baseline=True,
                         include_ci=True)

In [ ]:
tt = visualizer.MediaSummary(pairs[(geo,1)][1])
tt.plot_channel_contribution_area_chart()

In [ ]:
tt.plot_spend_vs_contribution()

In [ ]:
visualizer.MediaEffects(pairs[(geo,1)][1],use_kpi=True).plot_response_curves()

In [ ]:
dd = ['2025-09-01']

for d in dd:
    an = analyzer.Analyzer(pairs[(geo,1)][1])
    tbl_final = pd.DataFrame()
    obj = an.summary_metrics(aggregate_geos=False, aggregate_times=False, use_kpi=True)
    channels_used = list(pd.Index(obj.coords["channel"].values)[:-1])
    geos = list(pd.Index(obj.coords["geo"].values))
    dates = list(pd.Index(obj.coords["time"].values))
    for geo_target in geos:        # ← нужное GEO

        # --- 2) тензор помесячных вкладов (включаем baseline, чтобы можно было и с ним, и без него) ---
        ts = an.incremental_outcome(
            use_posterior=True, use_kpi=True,
            aggregate_times=False,          # помесячно
            aggregate_geos=False,           # по каждому GEO
        )  # ожидаем (chains, draws, G, T, K)
        
        arr = ts.numpy() if hasattr(ts, "numpy") else np.asarray(ts)
        arr_mean = arr.mean(axis=(0,1))                # → (G, T, K)
        
        # индексы
        gidx = geos.index(geo_target)
        G, T, K = arr_mean.shape
        assert T == len(dates), f"Длины дат не совпали: {T} vs {len(dates)}"
        assert K == len(channels_used), f"Длины каналов не совпали: {K} vs {len(channels_used)}"
        
        # --- 3) матрица вкладов для GEO и доли ---
        mat = arr_mean[gidx, :, :]                     # (T, K)
        tot_all = mat.sum(axis=1, keepdims=True)
        
   
        # доли всех каналов (включая baseline)
        share_all = np.where(tot_all > 0, mat / tot_all, np.nan) * 100.0
        tbl_all = pd.DataFrame(share_all, index=pd.to_datetime(dates), columns=channels_used)
        tbl_all.index.name = "date"
        tbl_all = tbl_all.round(2)
        tbl_all['geo']=geo_target
        tbl_all["date"] = tbl_all.index 
        # доли только платных каналов (нормируем на сумму по платным)
        share_paid = np.where(tot_all > 0, mat / tot_all, np.nan) * 100.0
        tbl_paid = pd.DataFrame(share_paid, index=pd.to_datetime(dates),
                                columns=channels_used)
        tbl_paid.index.name = "date"
        tbl_paid = tbl_paid.round(2)
        
        # --- 4) показать обе таблицы ---
        #print(f"ГЕО: {geo_target}")
        tbl_final = pd.concat([tbl_final, tbl_all], ignore_index=True)
    print(tbl_final.geo.unique())

In [ ]:
columns = list(tbl_final.columns[-2:])
columns += list(tbl_final.columns[:-2])
final_table = tbl_final[columns]
final_table['geo'] = geo 

In [ ]:
# saving data
with pd.ExcelWriter(f"shares_26_05_{geo}.xlsx") as writer:
    final_table.to_excel(writer, sheet_name=f"{geo}", index=False)

# Analysis
## How indicators are calculated on the channel quality graph

The graph is used to visually check the consistency of the channel between:
- the actual budget allocation,
- the channel's contribution according to the MMM,
- organic dynamics.

---

## 1. The observed share of expenses
$$
s_t = \frac{\text{Spend}_{c,t}}{\sum_j \text{Spend}_{j,t}}
$$

---

## 2. Central Line (EWMA of expenses share)
EWMA is used to smooth out noise and approximate the adstock effect:
$$
\text{center}_t = \alpha s_t + (1-\alpha)\text{center}_{t-1},
\qquad
\alpha = \frac{2}{k+1}
$$

где  
- $s_t$ — observed share of channel expenses,
- $k$ — span EWMA (5 months),  
- $\text{center}_0 = s_0$.

---

## 3. Confidence bands (EWMA-based)
Для классов **B / C / D** зона строится вокруг EWMA:
$$
[\;L\cdot\text{center}_t,\; U\cdot\text{center}_t\;]
$$

| Класс | Интерпретация | $[L,U]$ |
|------|---------------|---------|
| B | Канал эффективнее среднего | [1.10, 1.60] |
| C | Канал согласован | [0.85, 1.25] |
| D | Канал слабый | [0.40, 0.90] |

Зона **не отображается**, если $\text{center}_t < 3\%$.

---

## 4. p_stable

$p_{\text{stable}} = \Pr\big(\text{sign}(\rho_t) = \text{sign}(\rho^*)\big)$

Где $\rho_t$ — rolling-corr между Spend и $Organics_{t+k^*}$ (окно `WINDOW_ROLL`).
Интерпретация: доля окон, где знак корреляции совпадает со знаком $\rho^*$.
Если окон недостаточно — метрика `NA`.

---

## 4. Корреляция с Organics (|ρ*|)
Используется максимальная по лагам корреляция:
$$
\rho^* = \max_{l \in [0,4]} \left| \text{corr}\big(s_{t-l}, Organics_t\big) \right|
$$

Интерпретация:
- $|\rho^*| \ge 0.5$ — сильная связь,
- $0.3 \le |\rho^*| < 0.5$ — умеренная связь,
- $|\rho^*| < 0.3$ — слабая связь.

---

## 5. Lag-score (k*)
$$
k^* = \arg\max_{l \in [0,4]} \left| \text{corr}\big(s_{t-l}, Organics_t\big) \right|
$$

Показывает задержку реакции Organics на изменения канала.

---

## 6. HR (Hit Rate)
Доля периодов, когда изменения доли канала и Organics имеют одинаковый знак:
$$
HR = \frac{1}{T}\sum_t \mathbb{1}
\left(
\text{sign}(\Delta s_{t-k^*}) = \text{sign}(\Delta Organics_t)
\right)
$$

Интерпретация:
- $HR > 0.6$ — стабильная направленность эффекта,
- $HR \approx 0.5$ — слабая согласованность.

---

## 7. β (направление эффекта)
Знак робастного наклона:
$$
\beta = \text{sign}\Big(
\text{median}_t\frac{\Delta Organics_t}{\Delta s_{t-k^*}}
\Big)
$$

- $\beta=+$ — рост доли канала сопровождается ростом Organics,
- $\beta=-$ — рост доли канала сопровождается падением Organics.

---

## 8. Класс канала (A / B / C / D)

Класс канала определяется по совокупности метрик, а не только по величине корреляции.

### Классы каналов

**A — Low evidence**

- Недостаточно данных: менее **6 активных месяцев**

Интерпретация:  
данных недостаточно → **зоны доверия не строятся**, показываются только линии.

---

**B — High signal**

Все условия выполняются одновременно:

- $|\rho^*| > 0.5$
- $p_{\text{stable}} > 0.6$
- $HR > 0.7$
- не менее **6 активных месяцев**

Интерпретация:  
канал устойчиво и сильно связан с Organics →  
вклад MMM **ожидается выше** EWMA доли расходов.

---

**C — Aligned**

- все остальные случаи при **не менее 6 активных месяцах**

Интерпретация:  
умеренная или неоднозначная связь с Organics →  
вклад MMM **ожидается около** EWMA доли расходов.

---

**D — Low signal**

- $|\rho^*| < 0.2$
- не менее **6 активных месяцев**

Интерпретация:  
канал слабо связан с Organics →  
вклад MMM **ожидается ниже** EWMA доли расходов.

---

> Важно: высокая корреляция сама по себе **не означает** высокий класс.  
> Для класса B требуется подтверждённая **устойчивость во времени** и интерпретируемая динамика.

Класс определяет **ожидаемое положение MMM относительно EWMA**.


In [ ]:
EWMA_SPAN = 5
CENTER_MIN_SHARE = 0.03
COLLIN_THRESH = 0.6

BAND_BY_CLASS = {
    "B": (1.10, 1.60),
    "C": (0.85, 1.25),
    "D": (0.40, 0.90),
}


In [ ]:
def to_yyyymm(dt):
    return pd.to_datetime(dt).dt.strftime("%Y%m").astype(int)

def safe_corr(x, y, min_n=MIN_N_CORR):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < min_n:
        return np.nan, int(m.sum())
    if np.nanstd(x[m]) == 0 or np.nanstd(y[m]) == 0:
        return np.nan, int(m.sum())
    return float(np.corrcoef(x[m], y[m])[0, 1]), int(m.sum())

def fmt_num(x, nd=2):
    return f"{x:.{nd}f}" if np.isfinite(x) else "NA"

def fmt_beta(x):
    if not np.isfinite(x):
        return "NA"
    if x > 0:
        return "+"
    if x < 0:
        return "-"
    return "0"


In [ ]:
# HR and beta are for spends and organics (not for spends share and organics)

dfk = df[df["geo"] == GEO].copy()
dfk["time"] = pd.to_datetime(dfk["time"])
dfk = dfk.sort_values("time").reset_index(drop=True)
dfk["yyyymm"] = to_yyyymm(dfk["time"])

spend_cols = [c for c in dfk.columns if c.endswith("_spend")]
dfk["spend_total"] = dfk[spend_cols].sum(axis=1)

share_df = dfk[spend_cols].div(dfk["spend_total"].replace(0, np.nan), axis=0)

model = tbl_final
model['geo'] = geo
if "geo" in model.columns:
    model = model[model["geo"] == GEO].copy()
model = model.rename(columns={"date": "yyyymm"})
model["yyyymm"] = dfk["yyyymm"]
model_channels = [c for c in model.columns if c not in ["geo", "yyyymm"]]
for c in model_channels:
    model[c] = model[c].astype(float) / 100.0

rows = []
plot_df_list = []

for spend_col, model_col in SPEND_TO_MODEL.items():
    if spend_col not in spend_cols:
        continue

    s = share_df[spend_col].astype(float)
    sp = dfk[spend_col].astype(float)
    organics = dfk[TARGET].astype(float)

    n_active = int((dfk[spend_col].fillna(0) > 0).sum())

    corr_by_lag, n_by_lag = {}, {}
    for lag in LAGS:
        organics_shift = organics.shift(-lag)
        v, n_used = safe_corr(sp.values, organics_shift.values, min_n=MIN_N_CORR)
        corr_by_lag[lag] = v
        n_by_lag[lag] = n_used

    finite_lags = [l for l in LAGS if np.isfinite(corr_by_lag[l])]
    if finite_lags:
        k_star = max(finite_lags, key=lambda l: abs(corr_by_lag[l]))
        rho_star = corr_by_lag[k_star]
    else:
        k_star = np.nan
        rho_star = np.nan
    rho_star_abs = abs(rho_star) if np.isfinite(rho_star) else np.nan

    if np.isfinite(rho_star) and rho_star != 0 and np.isfinite(k_star):
        lag = int(k_star)
        organics_shift = organics.shift(-lag)
        active_mask = sp.fillna(0) > 0
        rho_t = rolling_corr_filtered(
            x=sp.values,
            y=organics_shift.values,
            window=WINDOW_ROLL,
            min_n=MIN_N_ACTIVE_IN_WINDOW,
            active_mask=active_mask.values,
        )
        valid = np.isfinite(rho_t) & (np.sign(rho_t) != 0)
        p_stable = float((np.sign(rho_t[valid]) == np.sign(rho_star)).mean()) if valid.any() else np.nan
    else:
        p_stable = np.nan

    if np.isfinite(k_star):
        lag = int(k_star)
        organics_shift = organics.shift(-lag)
    else:
        organics_shift = pd.Series(np.nan, index=organics.index)

    dsp = sp.diff()
    dorganics = organics_shift.diff()
    valid = np.isfinite(dsp) & np.isfinite(dorganics) & (np.abs(dsp) > 1e-9) & (np.abs(dorganics) > 1e-9)
    if valid.any():
        HR = float((np.sign(dsp[valid]) == np.sign(dorganics[valid])).mean())
        ratio = (dorganics / dsp).where(valid)
        beta_val = np.nanmedian(ratio.values)
        beta = float(np.sign(beta_val)) if np.isfinite(beta_val) else np.nan
    else:
        HR = np.nan
        beta = np.nan

    if n_active < 6:
        channel_class = "A"
    elif np.isfinite(rho_star) and ((rho_star) > 0.5) and (HR >= 0.5):#and (p_stable >= 0.6) and (HR >= 0.5):
        channel_class = "B"
    elif np.isfinite(rho_star) and (abs(rho_star) < 0.3):
        channel_class = "D"
    else:
        channel_class = "C"

    center = s.ewm(span=EWMA_SPAN, adjust=False).mean()
    band_lo = np.full_like(center.values, np.nan, dtype=float)
    band_hi = np.full_like(center.values, np.nan, dtype=float)
    if channel_class in BAND_BY_CLASS:
        lo_mul, hi_mul = BAND_BY_CLASS[channel_class]
        band_lo = center.values * lo_mul
        band_hi = center.values * hi_mul
        band_lo = np.where(center.values < CENTER_MIN_SHARE, np.nan, band_lo)
        band_hi = np.where(center.values < CENTER_MIN_SHARE, np.nan, band_hi)

    tmp = pd.DataFrame({
        "yyyymm": dfk["yyyymm"].values,
        "observed": s.values,
        "center": center.values,
        "band_lo": band_lo,
        "band_hi": band_hi,
        "organics": organics.values,
    })
    tmp["spend_col"] = spend_col
    tmp["model_col"] = model_col
    tmp["class"] = channel_class
    tmp["rho_star"] = rho_star
    tmp["k_star"] = k_star
    tmp["p_stable"] = p_stable
    tmp["HR"] = HR
    tmp["beta"] = beta

    if model_col in model.columns:
        tmp = tmp.merge(model[["yyyymm", model_col]], on="yyyymm", how="left")
        tmp = tmp.rename(columns={model_col: "model"})
    else:
        tmp["model"] = np.nan

    plot_df_list.append(tmp)

    rows.append({
        "spend_col": spend_col,
        "model_col": model_col,
        "class": channel_class,
        "n_active_months": n_active,
        "rho_star": rho_star,
        "rho_star_abs": rho_star_abs,
        "k_star": k_star,
        "p_stable": p_stable,
        "HR": HR,
        "beta": beta
    })

qual = pd.DataFrame(rows)

plot_df = pd.concat(plot_df_list, ignore_index=True) if plot_df_list else pd.DataFrame()

qual_display = qual.copy()
for c in ["rho_star", "rho_star_abs", "p_stable", "HR", "collin_max"]:
    if c in qual_display.columns:
        qual_display[c] = qual_display[c].round(2)
if "k_star" in qual_display.columns:
    qual_display["k_star"] = qual_display["k_star"].apply(lambda x: int(x) if np.isfinite(x) else np.nan)
if "beta" in qual_display.columns:
    qual_display["beta"] = qual_display["beta"].apply(lambda x: "+" if np.isfinite(x) and x > 0 else "-" if np.isfinite(x) and x < 0 else np.nan)

qual_display = qual_display.sort_values(["class", "rho_star_abs"], ascending=[True, False])
qual_display

In [ ]:
def plot_channel_quality(plot_df, spend_col, max_points=None):
    d = plot_df[plot_df["spend_col"] == spend_col].copy()
    if d.empty:
        return

    d["time"] = pd.to_datetime(d["yyyymm"].astype(str) + "01", format="%Y%m%d")
    if max_points is not None and len(d) > max_points:
        d = d.tail(max_points)

    fig, ax = plt.subplots(figsize=(12, 4))

    has_band = d["band_lo"].notna().any()
    if has_band:
        ax.fill_between(d["time"], d["band_lo"] * 100, d["band_hi"] * 100, alpha=0.2, label="Band (EWMA)")

    ax.plot(d["time"], d["observed"] * 100, label="Observed spend share (%)")
    ax.plot(d["time"], d["center"] * 100, linestyle="--", label="EWMA center (%)")
    if d["model"].notna().any():
        ax.plot(d["time"], d["model"] * 100, label="MMM share (%)")

    ax2 = None
    if "organics" in d.columns and d["organics"].notna().any():
        ax2 = ax.twinx()
        ax2.plot(d["time"], d["organics"], color="0.7", linestyle="--", linewidth=1.5, label="Organics")
        ax2.set_ylabel("Organics")

    if has_band and d["model"].notna().any():
        oob = d[
            d["model"].notna()
            & d["band_lo"].notna()
            & ((d["model"] < d["band_lo"]) | (d["model"] > d["band_hi"]))
        ]
        if not oob.empty:
            ax.scatter(oob["time"], oob["model"] * 100, s=30, marker="x", label="MMM out of band")

    first = d.iloc[0]
    k_star = int(first["k_star"]) if np.isfinite(first["k_star"]) else "NA"
    title = (
        f"{first['spend_col']} ↔ {first['model_col']} | class={first['class']}"
    )

    info_txt = (
        f"|ρ*|={fmt_num(abs(first['rho_star']))}\n k*={k_star}\n p_std={fmt_num(first['p_stable'])}\n"
        f"HR={fmt_num(first['HR'])}\n beta={fmt_beta(first['beta'])}"
    )

    ax.text(
        1.15, 0.45,   
        info_txt,
        transform=ax.transAxes,
        fontsize=12,
        va="top",
        ha="left",
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="lightgrey",
            alpha=0.9
        )
    )

    ax.set_title(title)
    ax.set_ylabel("Share (%)")
    ax.set_ylim(bottom=0)
    format_time_axis(ax, interval=6, rotation=45, ha="right")
    if ax2 is not None:
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, bbox_to_anchor=(1.1, 1), loc="upper left")
    else:
        ax.legend(bbox_to_anchor=(1.1, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

if not plot_df.empty:
    for sc in [c for c in SPEND_TO_MODEL.keys() if c in plot_df["spend_col"].unique()]:
        plot_channel_quality(plot_df, sc)

In [ ]:
# HR and beta are for spends and organics (not for spends share and organics) for the last year

dfk = df[df["geo"] == GEO].copy()
dfk["time"] = pd.to_datetime(dfk["time"])

# taking only one last year
cutoff = pd.Timestamp.now().normalize().replace(day=1) - pd.DateOffset(months=12)
dfk = dfk[dfk['time'] >= cutoff]

dfk = dfk.sort_values("time").reset_index(drop=True)
dfk["yyyymm"] = to_yyyymm(dfk["time"])

spend_cols = [c for c in dfk.columns if c.endswith("_spend")]
dfk["spend_total"] = dfk[spend_cols].sum(axis=1)

share_df = dfk[spend_cols].div(dfk["spend_total"].replace(0, np.nan), axis=0)

model = tbl_final
model['geo'] = geo
if "geo" in model.columns:
    model = model[model["geo"] == GEO].copy()
model = model.rename(columns={"date": "yyyymm"})
model["yyyymm"] = dfk["yyyymm"]
model_channels = [c for c in model.columns if c not in ["geo", "yyyymm"]]
for c in model_channels:
    model[c] = model[c].astype(float) / 100.0

rows = []
plot_df_list = []

for spend_col, model_col in SPEND_TO_MODEL.items():
    if spend_col not in spend_cols:
        continue

    s = share_df[spend_col].astype(float)
    sp = dfk[spend_col].astype(float)
    organics = dfk[TARGET].astype(float)

    n_active = int((dfk[spend_col].fillna(0) > 0).sum())

    corr_by_lag, n_by_lag = {}, {}
    for lag in LAGS:
        organics_shift = organics.shift(-lag)
        v, n_used = safe_corr(sp.values, organics_shift.values, min_n=MIN_N_CORR)
        corr_by_lag[lag] = v
        n_by_lag[lag] = n_used

    finite_lags = [l for l in LAGS if np.isfinite(corr_by_lag[l])]
    if finite_lags:
        k_star = max(finite_lags, key=lambda l: abs(corr_by_lag[l]))
        rho_star = corr_by_lag[k_star]
    else:
        k_star = np.nan
        rho_star = np.nan
    rho_star_abs = abs(rho_star) if np.isfinite(rho_star) else np.nan

    if np.isfinite(rho_star) and rho_star != 0 and np.isfinite(k_star):
        lag = int(k_star)
        organics_shift = organics.shift(-lag)
        active_mask = sp.fillna(0) > 0
        rho_t = rolling_corr_filtered(
            x=sp.values,
            y=organics_shift.values,
            window=WINDOW_ROLL,
            min_n=MIN_N_ACTIVE_IN_WINDOW,
            active_mask=active_mask.values,
        )
        valid = np.isfinite(rho_t) & (np.sign(rho_t) != 0)
        p_stable = float((np.sign(rho_t[valid]) == np.sign(rho_star)).mean()) if valid.any() else np.nan
    else:
        p_stable = np.nan

    if np.isfinite(k_star):
        lag = int(k_star)
        organics_shift = organics.shift(-lag)
    else:
        organics_shift = pd.Series(np.nan, index=organics.index)

    dsp = sp.diff()
    dorganics = organics_shift.diff()
    valid = np.isfinite(dsp) & np.isfinite(dorganics) & (np.abs(dsp) > 1e-9) & (np.abs(dorganics) > 1e-9)
    if valid.any():
        HR = float((np.sign(dsp[valid]) == np.sign(dorganics[valid])).mean())
        ratio = (dorganics / dsp).where(valid)
        beta_val = np.nanmedian(ratio.values)
        beta = float(np.sign(beta_val)) if np.isfinite(beta_val) else np.nan
    else:
        HR = np.nan
        beta = np.nan

    if n_active < 6:
        channel_class = "A"
    elif np.isfinite(rho_star) and (rho_star > 0.6) and (HR >= 0.5):#and (p_stable >= 0.6) and (HR >= 0.5):
        channel_class = "B"
    elif np.isfinite(rho_star) and (abs(rho_star) < 0.3):
        channel_class = "D"
    else:
        channel_class = "C"

    center = s.ewm(span=EWMA_SPAN, adjust=False).mean()
    band_lo = np.full_like(center.values, np.nan, dtype=float)
    band_hi = np.full_like(center.values, np.nan, dtype=float)
    if channel_class in BAND_BY_CLASS:
        lo_mul, hi_mul = BAND_BY_CLASS[channel_class]
        band_lo = center.values * lo_mul
        band_hi = center.values * hi_mul
        band_lo = np.where(center.values < CENTER_MIN_SHARE, np.nan, band_lo)
        band_hi = np.where(center.values < CENTER_MIN_SHARE, np.nan, band_hi)

    tmp = pd.DataFrame({
        "yyyymm": dfk["yyyymm"].values,
        "observed": s.values,
        "center": center.values,
        "band_lo": band_lo,
        "band_hi": band_hi,
        "organics": organics.values,
    })
    tmp["spend_col"] = spend_col
    tmp["model_col"] = model_col
    tmp["class"] = channel_class
    tmp["rho_star"] = rho_star
    tmp["k_star"] = k_star
    tmp["p_stable"] = p_stable
    tmp["HR"] = HR
    tmp["beta"] = beta

    if model_col in model.columns:
        tmp = tmp.merge(model[["yyyymm", model_col]], on="yyyymm", how="left")
        tmp = tmp.rename(columns={model_col: "model"})
    else:
        tmp["model"] = np.nan

    plot_df_list.append(tmp)

    rows.append({
        "spend_col": spend_col,
        "model_col": model_col,
        "class": channel_class,
        "n_active_months": n_active,
        "rho_star": rho_star,
        "rho_star_abs": rho_star_abs,
        "k_star": k_star,
        "p_stable": p_stable,
        "HR": HR,
        "beta": beta
    })

qual = pd.DataFrame(rows)
plot_df = pd.concat(plot_df_list, ignore_index=True) if plot_df_list else pd.DataFrame()

qual_display_one_year = qual.copy()
for c in ["rho_star", "rho_star_abs", "p_stable", "HR", "collin_max"]:
    if c in qual_display.columns:
        qual_display_one_year[c] = qual_display_one_year[c].round(2)
if "k_star" in qual_display.columns:
    qual_display_one_year["k_star"] = qual_display_one_year["k_star"].apply(lambda x: int(x) if np.isfinite(x) else np.nan)
if "beta" in qual_display.columns:
    qual_display_one_year["beta"] = qual_display_one_year["beta"].apply(lambda x: "+" if np.isfinite(x) and x > 0 else "-" if np.isfinite(x) and x < 0 else np.nan)

qual_display_one_year = qual_display_one_year.sort_values(["class", "rho_star_abs"], ascending=[True, False])
qual_display_one_year

## Final classes

In [ ]:
qual_display

In [ ]:
qual_display_one_year

In [ ]:
final_class = merge_classes(qual_display.rename(columns = {'spend_col':'index', 'class':'Class'}), qual_display_one_year.rename(columns = {'spend_col':'index', 'class':'Class'}))
final_class

In [ ]:
final_class_corr

In [ ]:
class_dict = final_class.set_index('index')['Class_Final'].to_dict()
class_dict

In [ ]:
# HR and beta are for spends and organics (not for spends share and organics)

dfk = df[df["geo"] == GEO].copy()
dfk["time"] = pd.to_datetime(dfk["time"])
dfk = dfk.sort_values("time").reset_index(drop=True)
dfk["yyyymm"] = to_yyyymm(dfk["time"])

spend_cols = [c for c in dfk.columns if c.endswith("_spend")]
dfk["spend_total"] = dfk[spend_cols].sum(axis=1)

share_df = dfk[spend_cols].div(dfk["spend_total"].replace(0, np.nan), axis=0)

model = tbl_final
model['geo'] = geo
if "geo" in model.columns:
    model = model[model["geo"] == GEO].copy()
model = model.rename(columns={"date": "yyyymm"})
model["yyyymm"] = dfk["yyyymm"]
model_channels = [c for c in model.columns if c not in ["geo", "yyyymm"]]
for c in model_channels:
    model[c] = model[c].astype(float) / 100.0

rows = []
plot_df_list = []

for spend_col, model_col in SPEND_TO_MODEL.items():
    if spend_col not in spend_cols:
        continue

    s = share_df[spend_col].astype(float)
    sp = dfk[spend_col].astype(float)
    organics = dfk[TARGET].astype(float)

    n_active = int((dfk[spend_col].fillna(0) > 0).sum())

    corr_by_lag, n_by_lag = {}, {}
    for lag in LAGS:
        organics_shift = organics.shift(-lag)
        v, n_used = safe_corr(sp.values, organics_shift.values, min_n=MIN_N_CORR)
        corr_by_lag[lag] = v
        n_by_lag[lag] = n_used

    finite_lags = [l for l in LAGS if np.isfinite(corr_by_lag[l])]
    if finite_lags:
        k_star = max(finite_lags, key=lambda l: abs(corr_by_lag[l]))
        rho_star = corr_by_lag[k_star]
    else:
        k_star = np.nan
        rho_star = np.nan
    rho_star_abs = abs(rho_star) if np.isfinite(rho_star) else np.nan

    if np.isfinite(rho_star) and rho_star != 0 and np.isfinite(k_star):
        lag = int(k_star)
        organics_shift = organics.shift(-lag)
        active_mask = sp.fillna(0) > 0
        rho_t = rolling_corr_filtered(
            x=sp.values,
            y=organics_shift.values,
            window=WINDOW_ROLL,
            min_n=MIN_N_ACTIVE_IN_WINDOW,
            active_mask=active_mask.values,
        )
        valid = np.isfinite(rho_t) & (np.sign(rho_t) != 0)
        p_stable = float((np.sign(rho_t[valid]) == np.sign(rho_star)).mean()) if valid.any() else np.nan
    else:
        p_stable = np.nan

    if np.isfinite(k_star):
        lag = int(k_star)
        organics_shift = organics.shift(-lag)
    else:
        organics_shift = pd.Series(np.nan, index=organics.index)

    dsp = sp.diff()
    dorganics = organics_shift.diff()
    valid = np.isfinite(dsp) & np.isfinite(dorganics) & (np.abs(dsp) > 1e-9) & (np.abs(dorganics) > 1e-9)
    if valid.any():
        HR = float((np.sign(dsp[valid]) == np.sign(dorganics[valid])).mean())
        ratio = (dorganics / dsp).where(valid)
        beta_val = np.nanmedian(ratio.values)
        beta = float(np.sign(beta_val)) if np.isfinite(beta_val) else np.nan
    else:
        HR = np.nan
        beta = np.nan

    channel_class = class_dict.get(spend_col, None)

    center = s.ewm(span=EWMA_SPAN, adjust=False).mean()
    band_lo = np.full_like(center.values, np.nan, dtype=float)
    band_hi = np.full_like(center.values, np.nan, dtype=float)
    if channel_class in BAND_BY_CLASS:
        lo_mul, hi_mul = BAND_BY_CLASS[channel_class]
        band_lo = center.values * lo_mul
        band_hi = center.values * hi_mul
        band_lo = np.where(center.values < CENTER_MIN_SHARE, np.nan, band_lo)
        band_hi = np.where(center.values < CENTER_MIN_SHARE, np.nan, band_hi)

    tmp = pd.DataFrame({
        "yyyymm": dfk["yyyymm"].values,
        "observed": s.values,
        "center": center.values,
        "band_lo": band_lo,
        "band_hi": band_hi,
        "organics": organics.values,
    })
    tmp["spend_col"] = spend_col
    tmp["model_col"] = model_col
    tmp["class"] = channel_class
    tmp["rho_star"] = rho_star
    tmp["k_star"] = k_star
    tmp["p_stable"] = p_stable
    tmp["HR"] = HR
    tmp["beta"] = beta

    if model_col in model.columns:
        tmp = tmp.merge(model[["yyyymm", model_col]], on="yyyymm", how="left")
        tmp = tmp.rename(columns={model_col: "model"})
    else:
        tmp["model"] = np.nan

    plot_df_list.append(tmp)

    rows.append({
        "spend_col": spend_col,
        "model_col": model_col,
        "class": channel_class,
        "n_active_months": n_active,
        "rho_star": rho_star,
        "rho_star_abs": rho_star_abs,
        "k_star": k_star,
        "p_stable": p_stable,
        "HR": HR,
        "beta": beta
    })

qual = pd.DataFrame(rows)
plot_df = pd.concat(plot_df_list, ignore_index=True) if plot_df_list else pd.DataFrame()

qual_display = qual.copy()
for c in ["rho_star", "rho_star_abs", "p_stable", "HR", "collin_max"]:
    if c in qual_display.columns:
        qual_display[c] = qual_display[c].round(2)
if "k_star" in qual_display.columns:
    qual_display["k_star"] = qual_display["k_star"].apply(lambda x: int(x) if np.isfinite(x) else np.nan)
if "beta" in qual_display.columns:
    qual_display["beta"] = qual_display["beta"].apply(lambda x: "+" if np.isfinite(x) and x > 0 else "-" if np.isfinite(x) and x < 0 else np.nan)

qual_display = qual_display.sort_values(["class", "rho_star_abs"], ascending=[True, False])
qual_display

In [ ]:
def plot_channel_quality(plot_df, spend_col, max_points=None):
    d = plot_df[plot_df["spend_col"] == spend_col].copy()
    if d.empty:
        return

    d["time"] = pd.to_datetime(d["yyyymm"].astype(str) + "01", format="%Y%m%d")
    if max_points is not None and len(d) > max_points:
        d = d.tail(max_points)

    fig, ax = plt.subplots(figsize=(12, 4))

    has_band = d["band_lo"].notna().any()
    if has_band:
        ax.fill_between(d["time"], d["band_lo"] * 100, d["band_hi"] * 100, alpha=0.2, label="Band (EWMA)")

    ax.plot(d["time"], d["observed"] * 100, label="Observed spend share (%)")
    ax.plot(d["time"], d["center"] * 100, linestyle="--", label="EWMA center (%)")
    if d["model"].notna().any():
        ax.plot(d["time"], d["model"] * 100, label="MMM share (%)")

    ax2 = None
    if "organics" in d.columns and d["organics"].notna().any():
        ax2 = ax.twinx()
        ax2.plot(d["time"], d["organics"], color="0.7", linestyle="--", linewidth=1.5, label="Organics")
        ax2.set_ylabel("Organics")

    if has_band and d["model"].notna().any():
        oob = d[
            d["model"].notna()
            & d["band_lo"].notna()
            & ((d["model"] < d["band_lo"]) | (d["model"] > d["band_hi"]))
        ]
        if not oob.empty:
            ax.scatter(oob["time"], oob["model"] * 100, s=30, marker="x", label="MMM out of band")

    first = d.iloc[0]
    k_star = int(first["k_star"]) if np.isfinite(first["k_star"]) else "NA"
    title = (
        f"{first['spend_col']} ↔ {first['model_col']} | class={first['class']}"
    )

    info_txt = (
        f"|ρ*|={fmt_num(abs(first['rho_star']))}\n k*={k_star}\n p_std={fmt_num(first['p_stable'])}\n"
        f"HR={fmt_num(first['HR'])}\n beta={fmt_beta(first['beta'])}"
    )

    ax.text(
        1.15, 0.45,   
        info_txt,
        transform=ax.transAxes,
        fontsize=12,
        va="top",
        ha="left",
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="lightgrey",
            alpha=0.9
        )
    )

    ax.set_title(title)
    ax.set_ylabel("Share (%)")
    ax.set_ylim(bottom=0)
    format_time_axis(ax, interval=6, rotation=45, ha="right")
    if ax2 is not None:
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, bbox_to_anchor=(1.1, 1), loc="upper left")
    else:
        ax.legend(bbox_to_anchor=(1.1, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

if not plot_df.empty:
    for sc in [c for c in SPEND_TO_MODEL.keys() if c in plot_df["spend_col"].unique()]:
        plot_channel_quality(plot_df, sc)


![](images/class_D.png)

![](images/class_C.png)

![](images/class_B.png)